# Train on a free Colab GPU

Accepts **either** kind of zip:

- A dataset already exported by `backend/scripts/export_for_colab.py`
  (has `data.yaml` inside), **or**
- A plain zip of your own photos with hand-drawn red outlines on them —
  the same format the app's "Import pre-annotated images" feature accepts.
  This notebook extracts the outlines itself, right here on the GPU
  machine, using the exact same code the app uses. No 30MB chat-upload
  limit either — Colab's upload cell handles much larger files fine.

You can also optionally continue from a previous `best.pt` instead of
starting from stock COCO weights each time (see the "Optional: continue
from checkpoint" cell below) — same idea as the desktop app's "Continue
from checkpoint" option on the Train tab.

**Before running:** `Runtime` menu → `Change runtime type` → select **T4 GPU** → Save.

Then set `CLASS_NAME` below if needed, and `Runtime` → `Run all`. When it
gets to the upload cells, upload your zip (either kind), and optionally a
previous `best.pt` to continue from.

At the end, `trained-model.zip` (containing `best.pt`) downloads automatically —
that's what you bring back to **Detect → Import a model** (desktop app or web app).

In [1]:
CLASS_NAME = "pin"  # only used if you upload a raw zip of red-annotated photos

In [2]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

Tesla T4, 15360 MiB


In [3]:
!pip install -q ultralytics opencv-python-headless pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 25.5 MB/s eta 0:00:00


Clone the app's repo (just to reuse its red-outline-extraction code — nothing
else from it runs here).

In [4]:
!rm -rf repo
!git clone -q --depth 1 -b claude/vision-model-object-detection-jbwh8s https://github.com/Robokks/object-detection-.git repo
import sys

sys.path.insert(0, "repo/backend")

In [6]:
from google.colab import files

print("Upload a dataset zip (from export_for_colab.py) OR a plain zip of your red-annotated photos:")
uploaded = files.upload()
zip_names = [n for n in uploaded if n.endswith(".zip")]
assert zip_names, "Expected a .zip file"
dataset_zip = zip_names[0]
print("Using:", dataset_zip)

Upload a dataset zip (from export_for_colab.py) OR a plain zip of your red-annotated photos:


Saving AI_IMAGES.zip to AI_IMAGES.zip
Using: AI_IMAGES.zip


Optionally continue fine-tuning from a **previous `best.pt`** (from an earlier Colab run, or exported from the desktop/web app) instead of starting over from stock COCO weights every time. When the upload dialog below appears, either pick a `.pt` file, or click **Cancel** to skip and start from stock weights.

In [7]:
from google.colab import files

print("Optional: upload a previous best.pt to continue from (Cancel to skip):")
try:
    ckpt_upload = files.upload()
except Exception:
    ckpt_upload = {}
checkpoint_names = [n for n in ckpt_upload if n.endswith(".pt")]
BASE_CHECKPOINT = checkpoint_names[0] if checkpoint_names else None
print("Continuing from:", BASE_CHECKPOINT) if BASE_CHECKPOINT else print("Starting from stock yolov8n-seg.pt")

Optional: upload a previous best.pt to continue from (Cancel to skip):


Saving best.pt to best.pt
Continuing from: best.pt


This cell figures out which kind of zip you uploaded and builds `dataset/`
either way. For a raw photo zip, it runs the same red-outline extraction +
red-line removal the app does, then an 80/20 train/val split.

In [8]:
import random
import shutil
from pathlib import Path

shutil.unpack_archive(dataset_zip, "raw")
raw_root = Path("raw")
existing_yaml = next(raw_root.rglob("data.yaml"), None)

dataset_dir = Path("dataset")
if dataset_dir.exists():
    shutil.rmtree(dataset_dir)

if existing_yaml is not None:
    print("Found data.yaml — this is a pre-built dataset export, using it as-is.")
    shutil.copytree(existing_yaml.parent, dataset_dir)
else:
    print(f'No data.yaml found — treating this as raw red-annotated photos, class "{CLASS_NAME}".')
    from app.services import red_import_service

    image_files = sorted(
        p for p in raw_root.rglob("*") if p.suffix.lower() in (".png", ".jpg", ".jpeg", ".bmp", ".webp")
    )
    assert image_files, "No image files found in the uploaded zip"
    print(f"Found {len(image_files)} image(s), extracting outlines…")

    random.seed(42)
    shuffled = image_files[:]
    random.shuffle(shuffled)
    val_count = max(1, int(len(shuffled) * 0.2)) if len(shuffled) > 1 else 0
    val_set = set(shuffled[:val_count])

    for split in ("train", "val"):
        (dataset_dir / "images" / split).mkdir(parents=True, exist_ok=True)
        (dataset_dir / "labels" / split).mkdir(parents=True, exist_ok=True)

    total_shapes = 0
    for path in image_files:
        split = "val" if path in val_set else "train"
        content = path.read_bytes()
        try:
            clean_bytes, shapes = red_import_service.extract_annotated_image(content, CLASS_NAME)
        except Exception as e:
            print(f"  {path.name}: skipped ({e})")
            continue
        if not shapes:
            print(f"  {path.name}: no outlines found, skipped")
            continue

        stem = path.stem
        out_img = dataset_dir / "images" / split / f"{stem}.png"
        out_img.write_bytes(clean_bytes)

        from PIL import Image
        import io

        with Image.open(io.BytesIO(clean_bytes)) as im:
            width, height = im.size

        lines = []
        for shape in shapes:
            coords = []
            for px, py in shape.points:
                coords.append(f"{min(max(px / width, 0.0), 1.0):.6f}")
                coords.append(f"{min(max(py / height, 0.0), 1.0):.6f}")
            lines.append("0 " + " ".join(coords))
        (dataset_dir / "labels" / split / f"{stem}.txt").write_text("\n".join(lines))
        total_shapes += len(shapes)
        print(f"  {path.name}: {len(shapes)} shape(s) -> {split}")

    (dataset_dir / "data.yaml").write_text(
        f"path: .\ntrain: images/train\nval: images/val\nnames:\n  0: {CLASS_NAME}\n"
    )
    print(f"Total shapes extracted: {total_shapes}")

!echo '--- dataset/data.yaml ---'; cat dataset/data.yaml
!echo '--- image counts ---'; find dataset/images -type f | wc -l

No data.yaml found — treating this as raw red-annotated photos, class "pin".
Found 84 image(s), extracting outlines…
  img1.PNG: 5 shape(s) -> train
  img10.PNG: 5 shape(s) -> train
  img11.PNG: 3 shape(s) -> train
  img12.PNG: 6 shape(s) -> train
  img13.PNG: 6 shape(s) -> train
  img14.PNG: 7 shape(s) -> train
  img15.PNG: 5 shape(s) -> train
  img16.PNG: 7 shape(s) -> train
  img17.PNG: 5 shape(s) -> train
  img18.PNG: 8 shape(s) -> train
  img19.PNG: 10 shape(s) -> train
  img2.PNG: 5 shape(s) -> train
  img20.PNG: 8 shape(s) -> train
  img21.PNG: 7 shape(s) -> train
  img22.PNG: 7 shape(s) -> train
  img23.PNG: 2 shape(s) -> val
  img24.PNG: 9 shape(s) -> train
  img25.PNG: 1 shape(s) -> train
  img26.PNG: 12 shape(s) -> train
  img27.PNG: 10 shape(s) -> val
  img28.PNG: 5 shape(s) -> val
  img29.PNG: 4 shape(s) -> train
  img3.PNG: 3 shape(s) -> train
  img30.PNG: 6 shape(s) -> train
  img31.PNG: 6 shape(s) -> train
  img32.PNG: 8 shape(s) -> train
  img33.PNG: 7 shape(s) -> val


## Train

Fine-tunes `yolov8n-seg` — the same architecture the app trains locally — starting from `BASE_CHECKPOINT` if you uploaded one above, otherwise from stock pretrained COCO weights (labels are polygon outlines, so the app always trains the segmentation variant, even for plain box labels).
Adjust `epochs`/`imgsz`/`batch` as needed; watch the `val` mask mAP in the output and stop early (interrupt the cell) if it plateaus.

In [11]:
from ultralytics import YOLO
from pathlib import Path

# Dynamically update data.yaml to correct the path for ultralytics
data_yaml_path = Path("dataset/data.yaml")
if data_yaml_path.exists():
    with open(data_yaml_path, 'r') as f:
        lines = f.readlines()

    # Replace 'path: .' with 'path: ./dataset'
    updated_lines = []
    for line in lines:
        if line.strip() == "path: .":
            updated_lines.append("path: ./dataset\n")
        else:
            updated_lines.append(line)

    with open(data_yaml_path, 'w') as f:
        f.writelines(updated_lines)

    print(f"Updated {data_yaml_path} with path: ./dataset")

model = YOLO(BASE_CHECKPOINT if BASE_CHECKPOINT else "yolov8n-seg.pt")
results = model.train(
    data=str(data_yaml_path),
    epochs=100,
    imgsz=640,
    batch=16,
    project="runs",
    name="train",
    exist_ok=True,
)

Updated dataset/data.yaml with path: ./dataset
Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=

In [12]:
# quick sanity check on the validation set
metrics = model.val()
print(metrics.seg.map, "(mask mAP50-95)")

Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8n-seg summary (fused): 86 layers, 3,258,259 parameters, 0 gradients, 11.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4917.7±1214.1 MB/s, size: 430.1 KB)
val: Scanning /content/dataset/labels/val.cache... 16 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 16/16 5.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.5it/s 0.6s
                   all         16         84      0.976      0.988      0.978      0.907      0.976      0.988      0.977      0.858
Speed: 0.4ms preprocess, 5.6ms inference, 0.0ms loss, 1.6ms postprocess per image
Results saved to /content/runs/segment/val
0.8579095284610245 (mask mAP50-95)


## Try the trained model + get full detection details as JSON

Optional — upload one or more test images to run the just-trained model on. Skip the upload (click **Cancel**) to use a few images from the validation split instead. Writes `colab-detections.json` with the same detail as the desktop app's Detect tab and `scripts/detect_to_json.py` (class, confidence, center position, orientation angle, left/right/top/bottom edge midpoints, and the bounding box for every detection) and downloads it.

In [13]:
from pathlib import Path
from google.colab import files

print("Optional: upload image(s) to test the trained model on (Cancel to use validation images instead):")
try:
    test_upload = files.upload()
except Exception:
    test_upload = {}

IMAGE_EXTS = (".png", ".jpg", ".jpeg", ".bmp", ".webp")
test_image_paths = [Path(n) for n in test_upload if Path(n).suffix.lower() in IMAGE_EXTS]
if not test_image_paths:
    # val split can end up empty for a very small dataset; fall back to train images then.
    for split in ("val", "train"):
        candidates = sorted(Path(f"dataset/images/{split}").glob("*"))
        if candidates:
            test_image_paths = candidates[:5]
            print(f"No images uploaded — using {len(test_image_paths)} image(s) from the {split} split instead.")
            break
else:
    print(f"Using {len(test_image_paths)} uploaded image(s).")

Optional: upload image(s) to test the trained model on (Cancel to use validation images instead):


No images uploaded — using 5 image(s) from the val split instead.


In [14]:
import json

from app.services import detect_service, report_service

DETECT_CONFIDENCE = 0.25  # adjust if you want more/fewer detections in the JSON

images_out = []
for img_path in test_image_paths:
    result = detect_service.run_detection_with_model(
        model, "segment", img_path.read_bytes(), DETECT_CONFIDENCE
    )
    images_out.append(
        {
            "image": str(img_path),
            "image_width": result.image_width,
            "image_height": result.image_height,
            "detections": [report_service.shape_to_report(b) for b in result.boxes],
        }
    )
    print(f"{img_path.name}: {len(result.boxes)} detection(s)")

output = {
    "model": "just-trained best.pt",
    "confidence_threshold": DETECT_CONFIDENCE,
    "images": images_out,
}
Path("colab-detections.json").write_text(json.dumps(output, indent=2))
print("Wrote colab-detections.json")

img23.png: 3 detection(s)
img27.png: 9 detection(s)
img28.png: 6 detection(s)
img33.png: 7 detection(s)
img37.png: 5 detection(s)
Wrote colab-detections.json


In [15]:
from google.colab import files

files.download("colab-detections.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
import shutil
from google.colab import files

# runs/train/ already has everything Ultralytics generates during training:
# weights/ (best.pt, last.pt), confusion matrix, PR/F1 curves, results.csv/png,
# args.yaml, train_batch*.jpg, val_batch0_labels/pred.jpg, etc. — zip the whole
# folder instead of just best.pt so all of that comes down too.
shutil.make_archive("training-run", "zip", "runs/segment/runs/train")
print("Zipped runs/segment/runs/train (weights + confusion matrix, PR curves, results.csv/png, etc.)")
files.download("training-run.zip")

Zipped runs/segment/runs/train (weights + confusion matrix, PR curves, results.csv/png, etc.)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>